# N069 · 树的递归与迭代遍历

**目标：** 把节点处理时机对应到前中后序。

**先修：** N009, N051, N062。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 树结构；递归栈；显式栈；遍历输出；N叉树推广。

## 从问题到算法

树遍历由“何时处理根节点”区分：前序在孩子之前，中序在两个孩子之间，后序在孩子之后。递归隐含的调用栈可改为显式栈：后序用展开标记，首次遇到节点先安排子树，再安排真正访问。N叉树前序同样在入栈时反转孩子顺序。

## 最小实现

**本章接口：** `preorder(root)`、`inorder(root)`、`postorder_iterative(root)`、`nary_preorder(root)`

In [1]:
class TreeNode:
    def __init__(self,val=0,left=None,right=None):
        self.val,self.left,self.right=val,left,right

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None: return None
    root=TreeNode(values[0]); queue=deque([root]); i=1
    while queue and i<len(values):
        node=queue.popleft()
        if values[i] is not None: node.left=TreeNode(values[i]); queue.append(node.left)
        i+=1
        if i<len(values):
            if values[i] is not None: node.right=TreeNode(values[i]); queue.append(node.right)
            i+=1
    return root

def preorder(root):
    if not root: return []
    answer=[]; stack=[root]
    while stack:
        node=stack.pop(); answer.append(node.val)
        if node.right: stack.append(node.right)
        if node.left: stack.append(node.left)
    return answer

def inorder(root):
    answer=[]; stack=[]; node=root
    while node or stack:
        while node: stack.append(node); node=node.left
        node=stack.pop(); answer.append(node.val); node=node.right
    return answer

def postorder_iterative(root):
    out=[]; stack=[(root,False)]
    while stack:
        node,expanded=stack.pop()
        if not node: continue
        if expanded: out.append(node.val)
        else:
            stack.append((node,True)); stack.append((node.right,False)); stack.append((node.left,False))
    return out

class NaryNode:
    def __init__(self,val,children=None): self.val=val; self.children=[] if children is None else children

def nary_preorder(root):
    out=[]; stack=[root] if root else []
    while stack:
        node=stack.pop(); out.append(node.val); stack.extend(reversed(node.children))
    return out

## 正确性、前提与复杂度

显式栈中的节点或展开标记表示尚未完成的递归调用。后入先出配合右孩子先入栈，使左子树先访问；展开标记只在两个子树处理之后弹出。

**代价：** O(n)时间；二叉树工作栈O(h)，输出O(n)。N叉树显式前序栈最坏O(n)。实现不依赖Python递归深度。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

r=tree_from_level([1,2,3,4,5])
show_table(['遍历','输出顺序'], [('前序',preorder(r)),('中序',inorder(r)),('后序',postorder_iterative(r))])

遍历,输出顺序
前序,"[1, 2, 4, 5, 3]"
中序,"[4, 2, 5, 1, 3]"
后序,"[4, 5, 2, 3, 1]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
r=tree_from_level([1,2,3,4,5])
assert preorder(r)==[1,2,4,5,3]
assert inorder(r)==[4,2,5,1,3]
assert postorder_iterative(r)==[4,5,2,3,1]
assert preorder(None)==inorder(None)==postorder_iterative(None)==[]
def reference(node):
    if not node: return [],[],[]
    a,b,c=reference(node.left); d,e,f=reference(node.right)
    return [node.val]+a+d,b+[node.val]+e,c+f+[node.val]
assert (preorder(r),inorder(r),postorder_iterative(r))==reference(r)
r=TreeNode(0); node=r
for i in range(1,1500): node.right=TreeNode(i); node=node.right
assert preorder(r)==inorder(r)==list(range(1500))
assert postorder_iterative(r)==list(range(1499,-1,-1))
n=NaryNode(1,[NaryNode(2),NaryNode(3,[NaryNode(4)])]); assert nary_preorder(n)==[1,2,3,4]
print('N069: 所有本章断言通过')

N069: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 每种遍历分别实现递归与迭代。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较N叉树的孩子展开顺序。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 144. Binary Tree Preorder Traversal（canonical）
- 94. Binary Tree Inorder Traversal（canonical）
- 145. Binary Tree Postorder Traversal（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。